In [ ]:
from pathlib import Path
import sys
import os
import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cmocean.cm as cmo
import cartopy.crs as ccrs
import matplotlib.dates as mdates
import cartopy.feature as cfeature
import xarray as xr
current_dir = Path.cwd()
src_path = Path(current_dir).parent / "src"
sys.path.insert(0, str(src_path))
from fetch_heincke_data import heincke_download_underway_data
from fetch_voto_data import glider_download_nrt_data, sailbuoy_download_nrt_data

from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from matplotlib_scalebar.scalebar import ScaleBar

### Heincke data

We get this from the Heincke data API (You can see the details in `sr/fetch_heincke_data.py`)

By default, it fetches the most recent 24 hours. In this example, I ask for all the data from August. If you request too many rows the API will send an error! Pulling < 3 months is usually fine though. You can always request multiple batches, save them and combine them.

**N.B.** If you get an error trying to re-download, just load your existing data with the cell below

@Callum, does not work for me at the moment: 
---> 47 df = df[['datetime',
     48          'vessel:heincke:trimble_5228k50585:longitude (mean) [°]',
     49          'vessel:heincke:trimble_5228k50585:latitude (mean) [°]', ]]
     50 df = df.rename({'vessel:heincke:trimble_5228k50585:longitude (mean) [°]': 'lon',
     51                 'vessel:heincke:trimble_5228k50585:latitude (mean) [°]': 'lat', }, axis=1)
...
   6249     raise KeyError(f"None of [{key}] are in the [{axis_name}]")
   6251 not_found = list(ensure_index(key)[missing_mask.nonzero()[0]].unique())
-> 6252 raise KeyError(f"{not_found} not in index")

In [ ]:


time_now = datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)

# df_heincke = heincke_download_underway_data(start=datetime.datetime(2025,8,15), end=time_now)

In [ ]:
# def read_heincke_data():
#     df = pd.read_csv("heincke_raw_data.csv", sep='\t', parse_dates=['datetime'])

#     df = df.rename({'vessel:heincke:trimble:longitude (mean) []': 'lon',
#                     'vessel:heincke:trimble:latitude (mean) []': 'lat', 
#                     'vessel:heincke:tsg:salinity (mean) [0/00]': 'salinity [PSU]',
#                     'vessel:heincke:tsg:sbe38:temperature (mean) [°C]': 'temperature [°C]',
#                     }, axis=1)
#     return df
# df_heincke = read_heincke_data()

In [ ]:
#df_heincke.keys()

### Glider data
You can pass any dataset id from the [VOTO ERDDAP](https://erddap.observations.voiceoftheocean.org/erddap/info/index.html)

There is a glider at the moment in the Skagerak, we also download the data from that.

In [ ]:
df_glider_SkaMix = glider_download_nrt_data(dataset_id="nrt_SEA044_M109").assign(dataset_id="nrt_SEA044_M109")

df_glider_Skagerak = glider_download_nrt_data(dataset_id="nrt_SEA069_M52").assign(dataset_id="nrt_SEA069_M52")




### Sailbuoy data

Very similar to glider data, just a different variable name for temperature (maybe we should standardise this upstream?)

In [ ]:
df_sailbuoy = sailbuoy_download_nrt_data()
print(df_sailbuoy.keys())

### All Positions in one plot, having Satellite Data as a background


Convert all datetime datetime arrays to the same format, posible nice more upstream?

In [ ]:
df_sailbuoy['datetime_utc'] = pd.to_datetime(df_sailbuoy['datetime'], utc=True).dt.tz_convert(None)
# df_heincke['datetime_utc'] = pd.to_datetime(df_heincke['datetime'], utc=True).dt.tz_convert(None)
df_glider_Skagerak['datetime_utc'] = pd.to_datetime(df_glider_Skagerak['datetime'], utc=True).dt.tz_convert(None)
df_glider_SkaMix['datetime_utc'] = pd.to_datetime(df_glider_SkaMix['datetime'], utc=True).dt.tz_convert(None)


Prepare different plot functions


In [ ]:
def plot_sailbuoy(fig, ax, df_sailbuoy, user_preferences_map):

    startdate = user_preferences_map["startdate"] 
    enddate = user_preferences_map["enddate"]
    vmin = user_preferences_map["vlim"][0]
    vmax = user_preferences_map["vlim"][1]
    
    df_sailbuoy_filtered = df_sailbuoy
    if startdate is not None:
        df_sailbuoy_filtered = df_sailbuoy_filtered[df_sailbuoy_filtered['datetime_utc'] >= startdate].reset_index(drop=True)
    if enddate is not None:
        df_sailbuoy_filtered = df_sailbuoy_filtered[df_sailbuoy_filtered['datetime_utc'] <= enddate].reset_index(drop=True)
   
        
    c = ax.scatter(df_sailbuoy_filtered.lon, df_sailbuoy_filtered.lat, c=df_sailbuoy_filtered['TEMP (degree_C)'], cmap=cmo.thermal, vmin=vmin, vmax= vmax,  rasterized=True, transform=ccrs.PlateCarree(), zorder = 3)
    ax.set(xlabel='longitude', ylabel='latitide')
    


    #subset where lat and lon are not nan
    mask = df_sailbuoy_filtered['lon'].notna() & df_sailbuoy_filtered['lat'].notna()
    df_sailbuoy_filtered = df_sailbuoy_filtered[mask]

    #show last position as marker
    index_last_position = np.argmax(df_sailbuoy_filtered['datetime_utc'])
    ax.scatter(df_sailbuoy_filtered.lon.iloc[index_last_position], df_sailbuoy_filtered.lat.iloc[index_last_position], c='red' ,label=f'Last position Sailbuoy {df_sailbuoy_filtered['datetime_utc'].iloc[index_last_position]}', marker = 'x', transform=ccrs.PlateCarree(), zorder = 4)
    ax.legend()
    if vmin is None and vmax is None:
        plt.colorbar(c, label='Temperature Sailbuoy [°C]')
    return c
    

In [ ]:
def plot_heincke(fig, ax, df_heincke, startdate = None, enddate = None, vmin=None, vmax=None):
    df_heincke_filtered = df_heincke
    if startdate is not None:
        df_heincke_filtered = df_heincke_filtered[df_heincke_filtered['datetime_utc'] >= startdate].reset_index(drop=True)
    if enddate is not None:
        df_heincke_filtered = df_heincke_filtered[df_heincke_filtered['datetime_utc'] <= enddate].reset_index(drop=True)
   
        
    c = ax.scatter(df_heincke_filtered.lon, df_heincke_filtered.lat, c=df_heincke_filtered['temperature [°C]'], cmap=cmo.thermal, vmin=vmin, vmax=vmax,  rasterized=True, transform=ccrs.PlateCarree(), zorder = 3)
    ax.set(xlabel='longitude', ylabel='latitide')
    


    #subset where lat and lon are not nan
    mask = df_heincke_filtered['lon'].notna() & df_heincke_filtered['lat'].notna()
    df_heincke_filtered = df_heincke_filtered[mask]

    #show last position as marker
    index_last_position = np.argmax(df_heincke_filtered['datetime_utc'])
    ax.scatter(df_heincke_filtered.lon.iloc[index_last_position], df_heincke_filtered.lat.iloc[index_last_position], c='cyan' ,label='Last position Heincke', marker = 'x', transform=ccrs.PlateCarree(), zorder = 4)
    ax.legend()
    if vmin is None and vmax is None:
        plt.colorbar(c, label='Temperature Heincke [°C]')
    return c
    

In [ ]:
def plot_glider(fig, ax, df_glider, user_preferences_map):
    startdate = user_preferences_map["startdate"] 
    enddate = user_preferences_map["enddate"]
    vmin = user_preferences_map["vlim"][0]
    vmax = user_preferences_map["vlim"][1]
    
    df_glider_filtered = df_glider[df_glider['depth (m)'] < 10]
    if startdate is not None:
        df_glider_filtered = df_glider_filtered[df_glider_filtered['datetime_utc'] >= startdate].reset_index(drop=True)
    if enddate is not None:
        df_glider_filtered = df_glider_filtered[df_glider_filtered['datetime_utc'] <= enddate].reset_index(drop=True)

    c = ax.scatter(df_glider_filtered.lon, df_glider_filtered.lat, c=df_glider_filtered['temperature (Celsius)'], cmap=cmo.thermal, vmin=vmin, vmax=vmax,  rasterized=True, transform=ccrs.PlateCarree(), zorder = 3)
    ax.set(xlabel='longitude', ylabel='latitide')

    #subset where lat and lon are not nan
    mask = df_glider_filtered['lon'].notna() & df_glider_filtered['lat'].notna()
    df_glider_filtered = df_glider_filtered[mask]

    #show last position as marker
    index_last_position = np.argmax(df_glider_filtered['datetime_utc'])
    if df_glider_filtered['dataset_id'].iloc[index_last_position] == 'nrt_SEA044_M109':
        color = 'blue'
    else:
        color = 'green'
    
    ax.scatter(df_glider_filtered.lon.iloc[index_last_position], df_glider_filtered.lat.iloc[index_last_position], c=color ,label=f'Last position Glider {df_glider_filtered['datetime_utc'].iloc[index_last_position]}', marker = 'x', transform=ccrs.PlateCarree(), zorder = 4)
    
    
    if vmin is None and vmax is None:
        plt.colorbar(c, label='Temperature Glider [°C]')
    return c


In [ ]:


def plot_geo_features(fig, ax, *, zoomed_dim_plot, scalebar_km: float = 50,
                           scalbar_location: str = "lower left") -> None:
    # Formatters for lat/lon labels
    ax.set_extent(zoomed_dim_plot, crs=ccrs.PlateCarree())

    #todo wieder rein?
    gl = ax.gridlines(color='tab:gray', alpha=0.5, linestyle='--', draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.xlabels_top = False
    gl.ylabels_right = False
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    
    land = cfeature.LAND.with_scale('50m')
    ax.add_feature(land, facecolor='#dce2dfff', zorder =3)
    
    add_scalebar(ax, scalebar_km=scalebar_km, location=scalbar_location)
    
    
def add_scalebar(ax, *, scalebar_km: float, location: str = "lower left") -> None:

    # Projected axes are typically in meters; one data unit ≈ 1 m.
    # Convert to km for display: 1 m = 0.001 km
    sb = ScaleBar(
        dx=0.001,               # km per data unit (meter)
        units="km",             # display in kilometers
        fixed_value=scalebar_km,
        location=location,
        box_alpha=0.3,
    )

    ax.add_artist(sb)

### Things to decide for the plot:


In [ ]:
zoomed_dim_plot = [7, 11.5, 56.5, 59]

plot_projection = ccrs.AzimuthalEquidistant(np.nanmean(zoomed_dim_plot[0:2]), np.nanmean(zoomed_dim_plot[2:4])) # Equal distance projection -> needs the center

user_preferences_map = {
    "cmap": cmo.thermal,
    "z_max": 50,
    "vlim" : [16, 18],
    "startdate" : time_now-datetime.timedelta(days = 3),
    "enddate" : time_now,
    "zoomed_dim_plot" : zoomed_dim_plot,
    "scalebar_km" : 50,
    "plot_projection": plot_projection
}


The "%matplotlib Widget" makes the plots interactive (and usually uglier), if you don't want that, just delete

In [ ]:
# %matplotlib Widget and much slower

Plot without background

In [ ]:


fig = plt.figure(figsize=(10, 10))
ax = plt.axes(projection=plot_projection)


plot_geo_features(fig, ax, zoomed_dim_plot= user_preferences_map['zoomed_dim_plot'], scalebar_km=user_preferences_map['scalebar_km'])
c_sailbuoy = plot_sailbuoy(fig, ax, df_sailbuoy, user_preferences_map)
# c_heincke = plot_heinke(fig, ax, df_heincke, startdate = startdate, enddate = enddate, vmin=vlim[0], vmax=vlim[1])
c_glider_Skagerak = plot_glider(fig, ax, df_glider_Skagerak,user_preferences_map)
c_glider_SkaMix = plot_glider(fig, ax, df_glider_SkaMix, user_preferences_map)

plt.colorbar(c_glider_Skagerak, ax=ax, label='Temperature [°C]')
plt.legend()
plt.show()
plt.close()


### Get Satelite in the background

Get the singlepass SST data, might not be working, if you don't get the mails

In [ ]:
ground_path = Path.cwd().parent / "data" / "sat_singlepass"

try: 
    current_dir = os.getcwd()
    src_path = Path(current_dir).parent / "src"
    sys.path.insert(0, str(src_path))
    import fetch_satellite_singlepass
    fetch_satellite_singlepass.fetch_metop_sst_attachments()
    case_sat_daten_singlepass = True
except ImportError as e:
    case_sat_daten_singlepass = False
    print("Module 'fetch_satellite_singlepass' could not be imported:", e)
    raise

except Exception as e:
    case_sat_daten_singlepass = False
    print("Satellite attachment fetch failed:", e)
    raise
    
print("case_sat_daten_singlepass =", case_sat_daten_singlepass)


In [ ]:
def plot_Sat_Singlepass(ds, zoomed_dim_plot, plot_projection, v_lim = [15, None], title=None):
    
    # Voto planned track
    lats = [57.9313, 58.0308]
    lons = [9.6077, 9.3833]

    data_var_name = "mcsst"   
    fig = plt.figure(figsize=(10, 8))

    ax = plt.axes(projection=plot_projection)

    im = ax.pcolormesh(ds['lon'], ds['lat'], ds[data_var_name][0,:,:],cmap = cmo.thermal, transform=ccrs.PlateCarree(), vmin=v_lim[0], vmax=v_lim[1], zorder = 1, rasterized=True)
    cbar = plt.colorbar(im)
    cbar.set_label(data_var_name+ '\°C') 
    

    if title:
        plt.title(title)

    plot_geo_features(fig, ax, zoomed_dim_plot = zoomed_dim_plot)
    ax.scatter(lons, lats, color="red", zorder=10, transform=ccrs.PlateCarree(), label = "planned VOTO devices track")

    if title:
        plt.savefig(f"figures_sat_singlepass/{title}.png")
    plt.show()
    plt.close(fig)

In [ ]:

def plot_Sat_Singlepass_and_devices(ds, user_preferences_map, title=None):
    print(user_preferences_map['zoomed_dim_plot'])
    zoomed_dim_plot = user_preferences_map['zoomed_dim_plot']
    plot_projection = user_preferences_map['plot_projection']
    v_lim = user_preferences_map['vlim']

    # Voto planned track
    lats = [57.9313, 58.0308]
    lons = [9.6077, 9.3833]

    data_var_name = "mcsst"   
    fig = plt.figure(figsize=(10, 8))

    ax = plt.axes(projection=plot_projection)

    c_glider_Skagerak = plot_glider(fig, ax, df_glider_Skagerak, user_preferences_map)
    c_glider_SkaMix= plot_glider(fig, ax, df_glider_SkaMix, user_preferences_map)
    c_sailbuoy= plot_sailbuoy(fig, ax, df_sailbuoy, user_preferences_map)
    #plt.colorbar(c_glider_Skagerak, ax=ax, label='Temperature [°C]')
    # plt.legend()

    im = ax.pcolormesh(ds['lon'], ds['lat'], ds[data_var_name][0,:,:],cmap = cmo.thermal, transform=ccrs.PlateCarree(), vmin=v_lim[0], vmax=v_lim[1], zorder = 1, rasterized=True)
    # cbar = plt.colorbar(im)
    # cbar.set_label(data_var_name+ '\°C') 
    
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Temperature [°C]")

    if title:
        plt.title(title)

    plot_geo_features(fig, ax, zoomed_dim_plot = zoomed_dim_plot)
    ax.scatter(lons, lats, color="red", zorder=10, transform=ccrs.PlateCarree(), label = "planned VOTO devices track")

    out_dir = Path("insitu_nrt_data_satellite_singlepass")
    out_dir.mkdir(parents=True, exist_ok=True)
    #  out_file = out_dir / f"{title}.png"
    
    
    
    ###### ugly ship track
    
    path_to_Heincke = r'C:\Users\boskamp\Downloads\Heincke_test_dship.dat'

    df_heincke = pd.read_csv(
        path_to_Heincke,
        sep=r"\s+",
        skiprows=3,
        names=["date", "time", "lat", "lon", "temp"],
        na_values=["#"],
        encoding="cp1252",          # <- add this
        encoding_errors="replace",  # optional: avoid crash on rare bytes
        engine="python"             # safer for regex separators on Windows files
    )

    vmin = user_preferences_map['vlim'][0]
    vmax = user_preferences_map['vlim'][1]
    ax.scatter(df_heincke['lon'], df_heincke['lat'], c=df_heincke['temp'], cmap=cmo.thermal, vmin=vmin, vmax= vmax,  rasterized=True, transform=ccrs.PlateCarree(), zorder = 3)

    #plot_heincke(fig, ax, df_heincke)


    if title:
        plt.savefig(title)
    plt.show()
    #plt.close(fig)





In [ ]:
from pathlib import Path

zoomed_dim_plot = user_preferences_map['zoomed_dim_plot'] # [lon_min, lon_max, lat_min, lat_max]
vlim = user_preferences_map['vlim']

lon_min, lon_max, lat_min, lat_max = zoomed_dim_plot



ground_path = Path.cwd().parent / "data" / "sat_singlepass"
print(ground_path)
print(Path(".."))
out_dir = Path("figures_sat_singlepass")
out_dir.mkdir(parents=True, exist_ok=True)


# loop for singlepass
for file in ground_path.glob("*.nc"):
    # #if file.endswith("_dm.nc"):
    if file.stem == r'01_202509082000_singlepass.nc':
        
        title = str(file.stem) + "-vlim" + str(vlim)+ "-zoomed_dim" + str(zoomed_dim_plot)
        if True:#os.path.exists(f"figures_sat_singlepass/{title}.png") == False:
            nc_path = os.path.join(ground_path, file)
            ds = xr.open_dataset(nc_path)
            ds_cut = ds.where(
            (ds.lon >= lon_min) & (ds.lon <= lon_max) &
            (ds.lat >= lat_min) & (ds.lat <= lat_max),
            drop=True)


            plot_Sat_Singlepass_and_devices(ds_cut, user_preferences_map, title=None)


In [ ]:
# %matplotlib Widget

In [ ]:
from pathlib import Path
user_preferences_map['zoomed_dim_plot'] = [8, 11, 56, 59]
zoomed_dim_plot = user_preferences_map['zoomed_dim_plot'] # [lon_min, lon_max, lat_min, lat_max]
user_preferences_map['vlim'] = [16, 18]
vlim = user_preferences_map['vlim']

lon_min, lon_max, lat_min, lat_max = zoomed_dim_plot



ground_path = Path.cwd().parent / "data" / "sat_singlepass"
print(ground_path)
print(Path(".."))
out_dir = Path("figures_of_the_day")
out_dir.mkdir(parents=True, exist_ok=True)
 


# loop for singlepass
for file in ground_path.glob("*.nc"):
    # #if file.endswith("_dm.nc"):
    if file.stem == r'03_202509081900_singlepass':

        out_fig =  out_dir / file.stem 

        #title = str(file.stem) + "-vlim" + str(vlim)+ "-zoomed_dim" + str(zoomed_dim_plot)
        if True: #os.path.exists(f"figures_sat_singlepass/{title}.png") == False:
            nc_path = os.path.join(ground_path, file)
            ds = xr.open_dataset(nc_path)
            ds_cut = ds.where(
            (ds.lon >= lon_min) & (ds.lon <= lon_max) &
            (ds.lat >= lat_min) & (ds.lat <= lat_max),
            drop=True)


            plot_Sat_Singlepass_and_devices(ds_cut, user_preferences_map, title=out_fig)
